# Encoder

In [1]:
import torch
import torch.nn as nn
import math


# ============================================================
# 1. SINUSOIDAL POSITIONAL ENCODING
# ============================================================

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=100):

        super().__init__()

        # [max_len, d_model]
        pe = torch.zeros(max_len, d_model)

        # [max_len, 1]
        position = torch.arange(
            0,
            max_len,
            dtype=torch.float
        ).unsqueeze(1)

        # Used inside:
        # 10000 ^ (-2i / d_model)
        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float
            )
            * (-math.log(10000.0) / d_model)
        )

        # Even dimensions:
        # PE(pos, 2i) = sin(...)
        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        # Odd dimensions:
        # PE(pos, 2i+1) = cos(...)
        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        # [1, max_len, d_model]
        # The 1 is for broadcasting across batch dimension.
        pe = pe.unsqueeze(0)

        # Store positional encoding as a buffer.
        # It is not a trainable parameter.
        self.register_buffer("pe", pe)

    def forward(self, x):

        # x shape:
        # [batch_size, sequence_length, d_model]

        T = x.size(1)

        # Add positional information
        return x + self.pe[:, :T]


# ============================================================
# 2. MULTI-HEAD SELF-ATTENTION
# ============================================================

class MultiHeadSelfAttention(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads

        # Dimension handled by each head
        #
        # Example:
        # d_model = 8
        # num_heads = 2
        #
        # head_dim = 8 / 2 = 4

        self.head_dim = d_model // num_heads

        # ----------------------------------------------------
        # Linear projections
        # ----------------------------------------------------

        self.W_q = nn.Linear(
            d_model,
            d_model
        )

        self.W_k = nn.Linear(
            d_model,
            d_model
        )

        self.W_v = nn.Linear(
            d_model,
            d_model
        )

        # Combines the outputs of all heads
        self.W_o = nn.Linear(
            d_model,
            d_model
        )

    def forward(self, x, padding_mask=None):

        # ----------------------------------------------------
        # x:
        #
        # [B, T, C]
        #
        # B = batch size
        # T = sequence length
        # C = d_model
        # ----------------------------------------------------

        B, T, C = x.shape

        # ----------------------------------------------------
        # STEP 1: Create Q, K, V
        # ----------------------------------------------------

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Shapes:
        #
        # Q = [B, T, C]
        # K = [B, T, C]
        # V = [B, T, C]

        # ----------------------------------------------------
        # STEP 2: Split into multiple heads
        # ----------------------------------------------------

        Q = Q.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        )

        # Before transpose:
        #
        # [B, T, num_heads, head_dim]

        # We want:
        #
        # [B, num_heads, T, head_dim]

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Example:
        #
        # B = 1
        # T = 4
        # num_heads = 2
        # head_dim = 4
        #
        # Q = [1, 2, 4, 4]

        # ----------------------------------------------------
        # STEP 3: Calculate QK^T
        # ----------------------------------------------------

        scores = Q @ K.transpose(-2, -1)

        # Shape:
        #
        # [B, num_heads, T, T]
        #
        # Example:
        #
        # [1, 2, 4, 4]

        # Each token compares itself against
        # every token in the sequence.

        # ----------------------------------------------------
        # STEP 4: Scale by sqrt(d_k)
        # ----------------------------------------------------

        scores = scores / math.sqrt(
            self.head_dim
        )

        # ----------------------------------------------------
        # STEP 5: Padding mask
        #
        # IMPORTANT:
        #
        # Encoder DOES NOT use causal masking.
        #
        # But we may have padding tokens.
        # Those should not receive attention.
        # ----------------------------------------------------

        if padding_mask is not None:

            # padding_mask:
            #
            # [B, T]
            #
            # True  = real token
            # False = PAD

            mask = padding_mask[:, None, None, :]

            scores = scores.masked_fill(
                ~mask,
                float("-inf")
            )

        # ----------------------------------------------------
        # STEP 6: Softmax
        # ----------------------------------------------------

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # Shape:
        #
        # [B, num_heads, T, T]

        # Every row now represents:
        #
        # "How much should this token
        #  pay attention to every other token?"

        # ----------------------------------------------------
        # STEP 7: Attention × V
        # ----------------------------------------------------

        out = attention_weights @ V

        # Shape:
        #
        # [B, num_heads, T, head_dim]

        # ----------------------------------------------------
        # STEP 8: Combine all heads
        # ----------------------------------------------------

        out = out.transpose(1, 2)

        # [B, T, num_heads, head_dim]

        out = out.contiguous().view(
            B,
            T,
            self.d_model
        )

        # [B, T, d_model]

        # ----------------------------------------------------
        # STEP 9: Final output projection
        # ----------------------------------------------------

        out = self.W_o(out)

        # [B, T, d_model]

        return out


# ============================================================
# 3. FEED-FORWARD NETWORK
# ============================================================

class FeedForward(nn.Module):

    def __init__(self, d_model, dropout=0.1):

        super().__init__()

        self.net = nn.Sequential(

            # Expand representation
            nn.Linear(
                d_model,
                4 * d_model
            ),

            # Non-linearity
            nn.GELU(),

            # Project back
            nn.Linear(
                4 * d_model,
                d_model
            ),

            nn.Dropout(dropout)
        )

    def forward(self, x):

        return self.net(x)


# ============================================================
# 4. ONE ENCODER BLOCK
# ============================================================

class EncoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        dropout=0.1
    ):

        super().__init__()

        # First LayerNorm
        self.norm1 = nn.LayerNorm(
            d_model
        )

        # Multi-head self-attention
        self.attention = MultiHeadSelfAttention(
            d_model,
            num_heads
        )

        # Second LayerNorm
        self.norm2 = nn.LayerNorm(
            d_model
        )

        # Feed-forward network
        self.ffn = FeedForward(
            d_model,
            dropout
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
        padding_mask=None
    ):

        # ====================================================
        # PART 1
        #
        # LayerNorm
        #     ↓
        # Self-Attention
        #     ↓
        # Residual
        # ====================================================

        attention_output = self.attention(
            self.norm1(x),
            padding_mask
        )

        x = x + self.dropout(
            attention_output
        )

        # ====================================================
        # PART 2
        #
        # LayerNorm
        #     ↓
        # FFN
        #     ↓
        # Residual
        # ====================================================

        ffn_output = self.ffn(
            self.norm2(x)
        )

        x = x + ffn_output

        return x


# ============================================================
# 5. COMPLETE ENCODER
# ============================================================

class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        num_layers,
        max_len,
        dropout=0.1
    ):

        super().__init__()

        self.d_model = d_model

        # ----------------------------------------------------
        # Token embedding
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # ----------------------------------------------------
        # Positional encoding
        # ----------------------------------------------------

        self.position_encoding = PositionalEncoding(
            d_model,
            max_len
        )

        self.dropout = nn.Dropout(
            dropout
        )

        # ----------------------------------------------------
        # STACK MULTIPLE ENCODER BLOCKS
        # ----------------------------------------------------

        self.blocks = nn.ModuleList([

            EncoderBlock(
                d_model,
                num_heads,
                dropout
            )

            for _ in range(num_layers)
        ])

        # ----------------------------------------------------
        # Final LayerNorm
        # ----------------------------------------------------

        self.final_norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        token_ids,
        padding_mask=None
    ):

        # ====================================================
        # token_ids:
        #
        # [B, T]
        # ====================================================

        # ----------------------------------------------------
        # STEP 1: Token embedding
        # ----------------------------------------------------

        x = self.token_embedding(
            token_ids
        )

        # [B, T, d_model]

        # ----------------------------------------------------
        # STEP 2: Positional encoding
        # ----------------------------------------------------

        x = self.position_encoding(x)

        x = self.dropout(x)

        # ----------------------------------------------------
        # STEP 3: Pass through every encoder block
        # ----------------------------------------------------

        for block in self.blocks:

            x = block(
                x,
                padding_mask
            )

        # ----------------------------------------------------
        # STEP 4: Final normalization
        # ----------------------------------------------------

        x = self.final_norm(x)

        # ----------------------------------------------------
        # FINAL OUTPUT
        #
        # [B, T, d_model]
        # ----------------------------------------------------

        return x


# ============================================================
# 6. TEST OUR ENCODER
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # Hyperparameters
    # --------------------------------------------------------

    vocab_size = 10

    d_model = 8

    num_heads = 2

    num_layers = 2

    max_len = 20

    # --------------------------------------------------------
    # Create model
    # --------------------------------------------------------

    model = Encoder(
        vocab_size=vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        num_layers=num_layers,
        max_len=max_len
    )

    # --------------------------------------------------------
    # Example token IDs
    #
    # Suppose:
    #
    # "I love deep learning"
    #
    # became:
    #
    # [2, 5, 7, 3]
    # --------------------------------------------------------

    token_ids = torch.tensor([
        [2, 5, 7, 3]
    ])

    # --------------------------------------------------------
    # Padding mask
    #
    # True  = real token
    # False = padding
    #
    # No padding in this example,
    # so everything is True.
    # --------------------------------------------------------

    padding_mask = torch.tensor([
        [True, True, True, True]
    ])

    # --------------------------------------------------------
    # Forward pass
    # --------------------------------------------------------

    output = model(
        token_ids,
        padding_mask
    )

    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print("Token IDs:")
    print(token_ids)

    print("\nInput shape:")
    print(token_ids.shape)

    print("\nEncoder output shape:")
    print(output.shape)

    print("\nEncoder output:")
    print(output)

Token IDs:
tensor([[2, 5, 7, 3]])

Input shape:
torch.Size([1, 4])

Encoder output shape:
torch.Size([1, 4, 8])

Encoder output:
tensor([[[-7.4663e-01, -1.2692e-01,  5.3100e-01,  1.2129e+00,  5.1719e-01,
           8.7526e-01, -2.1511e+00, -1.1175e-01],
         [ 1.9722e-01, -8.4881e-01,  9.4166e-01,  8.5925e-01, -1.7742e+00,
           8.2117e-01, -1.0400e+00,  8.4372e-01],
         [-4.8817e-01, -3.5015e-01,  2.3120e+00, -6.5176e-01, -1.2862e+00,
           4.6305e-01,  2.3416e-03, -1.1769e-03],
         [-6.0403e-01, -9.9533e-01,  9.4778e-01,  1.1386e-01, -1.7315e+00,
           7.8173e-01,  3.0351e-02,  1.4571e+00]]],
       grad_fn=<NativeLayerNormBackward0>)
